<a href="https://colab.research.google.com/github/alexandregbedo5-source/clickyX/blob/feature%2Fai-image-detector/ai-image-detector/training/colab/ai_detector_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ClickyX — Entraînement du détecteur d'images générées par IA

Notebook Colab complet : **données → entraînement CNN (PyTorch) → évaluation → export ONNX → calibration de la fusion → vérification du pipeline complet**.

Le résultat attendu à la fin :

| Artefact | Rôle |
|---|---|
| `model/detector.onnx` | CNN exporté (EfficientNet-B0 par défaut), consommé par ONNX Runtime dans ClickyX |
| `model/model_card.json` | Traçabilité : architecture, données, métriques, SHA-256 |
| `model/calibration.json` | Poids de la régression logistique (FFT, bruit, fusion) ajustés sur le jeu de validation |
| `runs/<run>/report.md` | Rapport de performances (global, par générateur, robustesse) |

> Runtime recommandé : **GPU T4/L4/A100**. `Exécution → Modifier le type d'exécution → GPU`.

## Mode d'emploi (première utilisation de Colab)

1. **Activer le GPU** : menu *Exécution → Modifier le type d'exécution → Accélérateur matériel : GPU (T4)* → *Enregistrer*.
2. **Exécuter les cellules dans l'ordre**, une par une : survoler la cellule et cliquer sur le bouton ▶ à gauche (ou `Maj+Entrée`). Attendre la coche verte à gauche avant de lancer la suivante.
3. Les réglages modifiables apparaissent **à droite** de certaines cellules (formulaires). Les valeurs par défaut fonctionnent : ne rien changer la première fois.
4. La cellule 3 demande l'autorisation d'accéder à Google Drive : cliquer *Se connecter à Google Drive* puis accepter. Seuls les résultats (quelques centaines de Mo) y sont écrits.
5. **Garder l'onglet ouvert** pendant toute la durée (≈ 1 h 30 à 2 h) : une session gratuite s'arrête après ~90 min sans activité. Si la session se coupe, relancer toutes les cellules depuis le début (les téléchargements terminés ne sont pas refaits).
6. À la fin (cellule 11), le navigateur télécharge trois fichiers : `detector.onnx`, `calibration.json`, `model_card.json`. Ils vont dans le dossier `model/` de la branche `feature/ai-image-detector`.

En cas d'erreur rouge sous une cellule : copier le message complet et le transmettre tel quel.

In [1]:
#@title 1. Environnement
import subprocess, sys, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout or "Pas de GPU détecté")
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

Tesla T4, 15360 MiB

torch 2.11.0+cu128 | cuda: True


In [2]:
#@title 2. Récupérer le code (branche feature/ai-image-detector)
REPO_URL = "https://github.com/alexandregbedo5-source/clickyX"  #@param {type:"string"}
BRANCH = "feature/ai-image-detector"  #@param {type:"string"}
import os, pathlib
if not pathlib.Path("clickyX").exists():
    !git clone --depth 1 --branch "$BRANCH" "$REPO_URL" clickyX
%cd clickyX
!pip -q install -r requirements.txt -r requirements-training.txt
!pip -q install datasets huggingface_hub matplotlib
!python -m pytest -q tests/test_frequency.py tests/test_noise.py tests/test_fusion_calibration.py

/content/clickyX
.....................                                                    [100%]


In [3]:
#@title 3. Espace de travail : données sur le disque Colab, résultats sur Google Drive (optionnel)
# Les jeux de données pèsent ~25 Go (AIGenImages2026 seul : archive 11 Go) : ils vont TOUJOURS sur le disque local
# de la session (rapide, ~80 Go disponibles) et ne rentreraient pas dans un Drive gratuit de 15 Go.
# Seuls les checkpoints, rapports et artefacts finaux (quelques centaines de Mo) vont sur Drive, pour survivre
# à une déconnexion. Décocher USE_DRIVE pour tout garder dans la session (perdu à la fermeture).
USE_DRIVE = True  #@param {type:"boolean"}
from pathlib import Path
DATA = Path("/content/work/data")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/clickyx-ai-detector")
else:
    WORK = Path("/content/work")
RUNS = WORK / "runs"
for d in (DATA, RUNS): d.mkdir(parents=True, exist_ok=True)
print("Données (disque de la session) :", DATA)
print("Résultats (checkpoints, rapports, artefacts) :", WORK)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Données (disque de la session) : /content/work/data
Résultats (checkpoints, rapports, artefacts) : /content/drive/MyDrive/clickyx-ai-detector


## 4. Données

Trois sources complémentaires (voir `docs/ai_detector.md` § Datasets) :

| Source | Contenu | Usage recommandé |
|---|---|---|
| **GenImage** (NeurIPS 2023) | 1,35 M IA (SD 1.4/1.5, Midjourney, ADM, GLIDE, Wukong, VQDM, BigGAN) vs ImageNet | Socle d'entraînement (diversité GAN + diffusion), protocole *cross-generator* |
| **Community Forensics** (CVPR 2025) | 2,7 M IA de 4 803 générateurs (version *Small* : 278 K + 278 K réelles) | Diversité massive de générateurs → généralisation |
| **AIGenImages2026** (WildFC, 2026) | 5 439 images de 19 modèles 2024–2025 (FLUX.2, GPT-Image 1.5, Imagen 4, Seedream 4.5, Midjourney v7…) | Jeu de **test hors distribution** (générateurs récents) et *fine-tuning* final léger |

Le notebook télécharge par défaut un sous-ensemble de **Community Forensics-Small** (réelles + IA, licences redistribuables) et **AIGenImages2026** (réservé à l'évaluation). GenImage (≈ 200 Go) se dépose manuellement dans `DATA/GenImage` si disponible.

Les images sont **matérialisées en dossiers** (`real/…`, `ai/<générateur>/…`) puis un **manifest CSV** unique est construit : c'est le format pivot de `training/data.py`.


In [ ]:
#@title 4a. Community Forensics-Small → dossiers (réelles + IA, par générateur)
MAX_PER_CLASS = 6000  #@param {type:"integer"}
import io, itertools
from collections import Counter
from datasets import load_dataset
from PIL import Image

out = DATA / "commfor_small"
if not (out / "_done").exists():
    ds = load_dataset("OwensLab/CommunityForensics-Small", split="train", streaming=True).shuffle(seed=0, buffer_size=2000)
    counts = Counter()
    for row in ds:
        label = int(row["label"])
        if counts[label] >= MAX_PER_CLASS:
            if all(counts[l] >= MAX_PER_CLASS for l in (0, 1)): break
            continue
        gen = (row.get("model_name") or "unknown").replace("/", "__") if label == 1 else "real"
        sub = out / ("ai" if label == 1 else "real") / (gen if label == 1 else "")
        sub.mkdir(parents=True, exist_ok=True)
        img = Image.open(io.BytesIO(row["image_data"])).convert("RGB")
        ext = ".jpg" if (row.get("format") or "").upper() in ("JPEG", "JPG") else ".png"
        img.save(sub / f"{row['image_name'].rsplit('.',1)[0]}{ext}", quality=95) if ext == ".jpg" else img.save(sub / f"{row['image_name'].rsplit('.',1)[0]}{ext}")
        counts[label] += 1
        if sum(counts.values()) % 500 == 0: print(dict(counts))
    (out / "_done").touch()
print("commfor_small prêt :", out)

Resolving data files:   0%|          | 0/186 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/186 [00:00<?, ?it/s]

{0: 500}
{0: 1000}


In [ ]:
#@title 4b. AIGenImages2026 (19 générateurs 2024–2025) — évaluation hors distribution (archive 11 Go, ~10 min)
import shutil, tarfile
from huggingface_hub import snapshot_download
aig = DATA / "aigenimages2026"
if not (aig / "_done").exists():
    local = snapshot_download(repo_id="pthan12/AIGenImages2026", repo_type="dataset", local_dir=str(aig / "_raw"))
    for tar in Path(local).glob("*.tar*"):
        print("extraction de", tar.name, "…")
        with tarfile.open(tar) as tf: tf.extractall(aig / "extracted")
    (aig / "_done").touch()
    shutil.rmtree(aig / "_raw", ignore_errors=True)  # l'archive n'est plus utile : 11 Go libérés
# Structure de l'archive : .../AIGenImages2026/{train,test}/1_fake/<images à plat>, générateur dans le nom de fichier.
fake_dirs = sorted(p for p in (aig / "extracted").rglob("1_fake") if p.is_dir())
for d in fake_dirs: print(f"{d.parent.name}/1_fake : {sum(1 for _ in d.iterdir())} fichiers")
assert fake_dirs, "Aucun dossier 1_fake trouvé : l'extraction a échoué, relancer la cellule."


In [ ]:
#@title 4c. Manifests : entraînement/validation (Community Forensics) et test (AIGenImages2026 + réelles réservées)
# Validation « générateurs jamais vus » : --group-by-generator réserve des générateurs entiers à la validation.
!python -m training.data build-manifest \
    --ai-dir "$DATA/commfor_small/ai" --real-dir "$DATA/commfor_small/real" \
    --val-fraction 0.15 --group-by-generator --seed 0 --out "$DATA/manifest_trainval.csv"
!python -m training.data summary --root "$DATA/manifest_trainval.csv" --layout manifest --split train
!python -m training.data summary --root "$DATA/manifest_trainval.csv" --layout manifest --split val

# Jeu de test hors distribution : toutes les IA d'AIGenImages2026 + les réelles réservées à la validation.
# Les images IA sont à plat dans <split>/1_fake/ ; le générateur est encodé dans le nom du fichier :
#   fal-ai_gpt-image-1.5_<uuid>_0.png → gpt-image-1.5     image_midjourneyv7_93.png → midjourneyv7
import csv, re
from collections import Counter
from training.data import list_images
UUID = r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}"
def generator_from_name(stem):
    s = re.sub(r"^(fal-ai|image)_", "", stem)
    s = re.sub(rf"_{UUID}(_\d+)?$", "", s)
    return re.sub(r"_\d+$", "", s) or "unknown"

rows = [{"path": str(p), "label": "1", "generator": generator_from_name(p.stem),
         "source": f"aigen2026/{d.parent.name}", "split": "test"}
        for d in fake_dirs for p in list_images(d)]
n_ai = len(rows)
rows += [{**r, "split": "test"} for r in csv.DictReader(open(DATA / "manifest_trainval.csv"))
         if r["label"] == "0" and r["split"] == "val"]
with open(DATA / "manifest_test.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=["path", "label", "generator", "source", "split"]); w.writeheader(); w.writerows(rows)
print(f"manifest_test.csv : {n_ai} IA (AIGenImages2026) + {len(rows) - n_ai} réelles (val Community Forensics)")
print("Générateurs :", dict(sorted(Counter(r["generator"] for r in rows if r["label"] == "1").items())))


## 5. Entraînement

Protocole (détaillé dans `docs/ai_detector.md`) :

* **EfficientNet-B0** pré-entraîné ImageNet, tête à 1 logit, `BCEWithLogits` ;
* crops **224×224 à résolution native** (pas de redimensionnement global) ;
* **échantillonnage équilibré** classes × générateurs (`WeightedRandomSampler`) ;
* augmentations de **dégradation** : JPEG (q 55–100, p=0,5), rééchantillonnage (×0,5–1,5, p=0,3), flou léger (p=0,1), flip ;
* AdamW, weight decay 0,05, cosinus + warmup, **EMA** des poids, label smoothing 0,02 ;
* sélection du meilleur checkpoint sur l'**AUC de validation** (générateurs jamais vus), early stopping.


In [ ]:
#@title 5. Lancer l'entraînement
ARCH = "efficientnet_b0"  #@param ["efficientnet_b0", "efficientnetv2_s", "convnext_tiny", "resnet50"]
EPOCHS = 8  #@param {type:"integer"}
BATCH = 64  #@param {type:"integer"}
LR = 3e-4  #@param {type:"number"}
RUN = RUNS / f"{ARCH}"
!python training/train.py --data-root "$DATA/manifest_trainval.csv" --layout manifest \
    --arch "$ARCH" --epochs $EPOCHS --batch-size $BATCH --lr $LR --amp --workers 4 \
    --freeze-backbone-epochs 1 --ema 0.999 --early-stopping-patience 3 \
    --out-dir "$RUN"

In [ ]:
#@title 5b. Courbes d'apprentissage
import json, matplotlib.pyplot as plt
hist = json.load(open(RUN / "history.json"))
ep = [h["epoch"] for h in hist]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ep, [h["train_loss"] for h in hist], marker="o"); ax[0].set_title("train loss"); ax[0].set_xlabel("époque")
ax[1].plot(ep, [h["val"]["auc"] for h in hist if "val" in h], marker="o", label="AUC")
ax[1].plot(ep, [h["val"]["balanced_accuracy"] for h in hist if "val" in h], marker="s", label="bal. acc")
ax[1].legend(); ax[1].set_title("validation (générateurs jamais vus)"); ax[1].set_xlabel("époque")
plt.tight_layout(); plt.show()

## 6. Évaluation

* **Validation** (Community Forensics, générateurs jamais vus) : métriques globales + par générateur.
* **Test hors distribution** (AIGenImages2026) : générateurs 2024–2025 jamais vus.
* **Robustesse** : JPEG q75/q50, redimensionnement ×0,5, flou σ=1.


In [ ]:
#@title 6. Évaluer le checkpoint
!python training/evaluate.py --checkpoint "$RUN/best.pt" --data-root "$DATA/manifest_trainval.csv" --layout manifest --split val \
    --robustness jpeg75 jpeg50 resize0.5 blur1.0 --out-dir "$RUN/eval_val"
!python training/evaluate.py --checkpoint "$RUN/best.pt" --data-root "$DATA/manifest_test.csv" --layout manifest --split test \
    --robustness jpeg75 --out-dir "$RUN/eval_test_aigen2026"
from IPython.display import Markdown, display
display(Markdown(open(RUN / "eval_val" / "report.md").read()))
display(Markdown(open(RUN / "eval_test_aigen2026" / "report.md").read()))

In [ ]:
#@title 7. Export ONNX + model card (vérification torch ↔ onnxruntime incluse)
VERSION = "1.0.0"  #@param {type:"string"}
!python training/export_onnx.py --checkpoint "$RUN/best.pt" --out model/detector.onnx --version "$VERSION" \
    --training-data "Community Forensics-Small (sous-ensemble équilibré), val = générateurs jamais vus"
!python -c "import json; print(json.dumps(json.load(open('model/model_card.json')), indent=2)[:1500])"
!ls -la model/

In [ ]:
#@title 8. Calibrer les modules FFT / bruit et la fusion sur la validation (scores hors-pli)
!python training/calibrate_fusion.py --data-root "$DATA/manifest_trainval.csv" --layout manifest --split val \
    --onnx model/detector.onnx --out model/calibration.json --features-csv "$RUN/calibration_features.csv" \
    --source-name "Community Forensics-Small val (générateurs jamais vus) — $VERSION"
import json; rep = json.load(open("model/calibration_report.json"))
print("AUC hors-pli  fft:", round(rep["module_auc_oof"]["fft_score"], 3), " noise:", round(rep["module_auc_oof"]["noise_score"], 3),
      " cnn:", round(rep.get("cnn_auc", float('nan')), 3), " fusion full:", round(rep["fusion_full_oof"]["auc"], 3))

In [ ]:
#@title 9. Pipeline complet (FFT + bruit + CNN + fusion) tel que servi par l'API — chiffres finaux
!python training/evaluate.py --full-pipeline --model-path model/detector.onnx --calibration-path model/calibration.json \
    --data-root "$DATA/manifest_test.csv" --layout manifest --split test --robustness jpeg75 resize0.5 \
    --out-dir "$RUN/eval_full_pipeline"
display(Markdown(open(RUN / "eval_full_pipeline" / "report.md").read()))

In [ ]:
#@title 10. Vérification de bout en bout : CLI + API (contrat POST /detect-ai-image)
import csv, json
from fastapi.testclient import TestClient
from ai_detector.server import create_app
sample = next(csv.DictReader(open(DATA / "manifest_test.csv")))["path"]
!python -m ai_detector info
!python -m ai_detector detect "$sample" --json
client = TestClient(create_app())
print(client.get("/health").json()["fusion_mode"])
print(json.dumps(client.post("/detect-ai-image", json={"image_path": sample, "include_details": False}).json(), indent=2))

In [ ]:
#@title 11. Sauvegarder les artefacts (Drive) et télécharger
import shutil
dest = WORK / "artifacts" / VERSION
dest.mkdir(parents=True, exist_ok=True)
for f in ("model/detector.onnx", "model/model_card.json", "model/calibration.json", "model/calibration_report.json"):
    shutil.copy(f, dest)
shutil.copytree(RUN, dest / "run", dirs_exist_ok=True)
print("Artefacts copiés dans", dest)
try:
    from google.colab import files
    files.download("model/detector.onnx"); files.download("model/calibration.json"); files.download("model/model_card.json")
except Exception as e:
    print("Téléchargement manuel :", e)

## 12. Intégration dans le dépôt

1. Copier `detector.onnx`, `model_card.json`, `calibration.json` dans `model/` de la branche `feature/ai-image-detector`.
2. Si `detector.onnx` dépasse ~50 Mo (ConvNeXt), utiliser Git LFS : `git lfs track "model/*.onnx"`.
3. Coller `runs/<run>/eval_*/report.md` dans `docs/ai_detector.md` § *Rapport de performances*.
4. Vérifier localement : `python -m ai_detector info` doit afficher `"fusion_mode": "full"` et `"trained": true`.
